### Time Series Trends & Anomalies

- The line chart above shows the monthly average temperature, with annotations for the warmest and coolest months.
- The bar chart displays monthly total precipitation, highlighting the peak rainy season.
- Comment on visible trends or anomalies in the markdown cell below after running the plots.

In [1]:
# Monthly total precipitation
monthly_precip = df_clean.groupby([df_clean['date'].dt.to_period('M')])['PRECTOTCORR'].sum()
monthly_precip.index = monthly_precip.index.to_timestamp()
plt.figure(figsize=(12,5))
plt.bar(monthly_precip.index, monthly_precip.values, width=20)
plt.title('Monthly Total Precipitation (PRECTOTCORR) in Ethiopia')
plt.ylabel('Precipitation (mm)')
plt.xlabel('Month')
plt.grid(True)
# Annotate peak rainy months
peak_rain = monthly_precip.idxmax()
plt.annotate(f'Peak Rain: {peak_rain.strftime("%b %Y")}', xy=(peak_rain, monthly_precip.max()), xytext=(peak_rain, monthly_precip.max()+10), arrowprops=dict(arrowstyle='->'))
plt.show()

NameError: name 'df_clean' is not defined

In [ ]:
# Monthly average T2M
monthly_t2m = df_clean.groupby([df_clean['date'].dt.to_period('M')])['T2M'].mean()
monthly_t2m.index = monthly_t2m.index.to_timestamp()
plt.figure(figsize=(12,5))
plt.plot(monthly_t2m, marker='o')
plt.title('Monthly Average Temperature (T2M) in Ethiopia')
plt.ylabel('Temperature (°C)')
plt.xlabel('Month')
plt.grid(True)
# Annotate warmest/coolest months
warmest = monthly_t2m.idxmax()
coolest = monthly_t2m.idxmin()
plt.annotate(f'Warmest: {warmest.strftime("%b %Y")}', xy=(warmest, monthly_t2m.max()), xytext=(warmest, monthly_t2m.max()+0.5), arrowprops=dict(arrowstyle='->'))
plt.annotate(f'Coolest: {coolest.strftime("%b %Y")}', xy=(coolest, monthly_t2m.min()), xytext=(coolest, monthly_t2m.min()-1), arrowprops=dict(arrowstyle='->'))
plt.show()

## 5. Time Series Analysis

We will plot monthly average temperature (T2M) and total precipitation (PRECTOTCORR), annotating the warmest/coolest and peak rainy months.

In [ ]:
# Export cleaned data
clean_path = 'data/ethiopia_clean.csv'
df_clean.to_csv(clean_path, index=False)
print(f"Cleaned data exported to {clean_path}")

In [ ]:
# Forward-fill weather variables
weather_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_cols] = df[weather_cols].ffill()
# Drop rows with >30% missing values
row_missing_pct = df.isna().mean(axis=1)
df_clean = df.loc[row_missing_pct <= 0.3].copy()
print(f"Rows dropped due to >30% missing: {(row_missing_pct > 0.3).sum()}")
df_clean.reset_index(drop=True, inplace=True)

### Outlier Handling Decision

- Outliers are flagged above for each variable. For this analysis, we will retain outliers unless they are clearly data errors, as extreme values may represent real climate events.
- For missing values, we will forward-fill weather variables. If more than 30% of a row is missing, we will drop that row.

In [ ]:
# Outlier detection for key variables
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_scores = np.abs(stats.zscore(df[z_cols], nan_policy='omit'))
outlier_flags = (z_scores > 3)
outlier_counts = pd.Series(outlier_flags.sum(axis=0), index=z_cols)
outlier_counts

## 4. Outlier Detection & Basic Cleaning

We will compute Z-scores for key variables, flag outliers (|Z| > 3), and decide how to handle them. We will also handle remaining missing values.

### Interpretation of Summary Statistics and Missing Values

- The summary statistics above provide an overview of the distribution of each numeric variable.
- The missing value report lists columns with missing data and their percentage. Columns with >5% missing values should be noted for potential impact on analysis.
- Duplicate rows found and dropped: check the variable `duplicates` above for the count.

In [ ]:
# Missing value report
missing_report = df.isna().sum().to_frame('missing_count')
missing_report['missing_pct'] = 100 * missing_report['missing_count'] / len(df)
missing_report = missing_report[missing_report['missing_count'] > 0]
missing_report

In [ ]:
# Replace -999 with NaN
missing_before = df.isna().sum()
df.replace(-999, np.nan, inplace=True)
missing_after = df.isna().sum()

# Check for duplicates
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

# Summary statistics
describe = df.describe()
describe

## 3. Summary Statistics & Missing-Value Report

We will replace all -999 values with NaN, check for duplicates, and generate summary statistics and missing value reports.

In [ ]:
# Load Ethiopia climate data
ethiopia_path = '../data/ethiopia.csv' if not os.path.exists('data/ethiopia.csv') else 'data/ethiopia.csv'
df = pd.read_csv(ethiopia_path)
df['Country'] = 'Ethiopia'
# Convert YEAR and DOY to datetime
df['date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['date'].dt.month
df.head()

## 2. Data Loading & Date Parsing

We will load the Ethiopia climate data, add a Country column, and convert YEAR and DOY to a proper datetime column. We will also extract the Month for seasonal analysis.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import os

# Ethiopia Climate Data EDA

This notebook performs exploratory data analysis (EDA), cleaning, and visualization for Ethiopia's climate dataset as part of the African Climate Trend Analysis challenge.

---

## 1. Import Required Libraries
We begin by importing the necessary Python libraries for data manipulation and visualization.

# Ethiopia Climate EDA\n
This notebook implements Task 2 for Ethiopia: profiling, cleaning, and exploratory analysis.

In [ ]:
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
from scipy.stats import zscore

In [1]:
country = 'Ethiopia'\n
df = pd.read_csv('../data/ethiopia.csv')\n
df['Country'] = country\n
df['DATE'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')\n
df['Month'] = df['DATE'].dt.month\n
df = df.replace(-999, np.nan)\n
duplicate_count = df.duplicated().sum()\n
df = df.drop_duplicates()\n
print('Duplicates found:', duplicate_count)\n
df.head()

SyntaxError: unexpected character after line continuation character (404784715.py, line 1)

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns\n
display(df[numeric_cols].describe())\n
missing = df.isna().sum().to_frame('missing_count')\n
missing['missing_pct'] = missing['missing_count'] / len(df) * 100\n
display(missing.sort_values('missing_pct', ascending=False))

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']\n
z_df = df[z_cols].copy()\n
z_df = z_df.fillna(z_df.median(numeric_only=True))\n
z_values = np.abs(zscore(z_df, nan_policy='omit'))\n
outlier_mask = (z_values > 3).any(axis=1)\n
print('Outlier rows (|Z| > 3):', int(outlier_mask.sum()))

In [ ]:
row_missing_ratio = df.isna().mean(axis=1)\n
df = df[row_missing_ratio <= 0.30].copy()\n
weather_cols = ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']\n
for col in weather_cols:\n
    if col in df.columns:\n
        df[col] = df[col].ffill()\n
df.to_csv('../data/ethiopia_clean.csv', index=False)\n
df.shape

In [ ]:
monthly_temp = df.resample('M', on='DATE')['T2M'].mean()\n
warmest = monthly_temp.idxmax()\n
coolest = monthly_temp.idxmin()\n
plt.figure(figsize=(12, 4))\n
monthly_temp.plot()\n
plt.scatter([warmest, coolest], [monthly_temp.max(), monthly_temp.min()], color=['red', 'blue'])\n
plt.title('Monthly Average T2M (Ethiopia)')\n
plt.ylabel('Temperature (°C)')\n
plt.show()

In [ ]:
monthly_prec = df.resample('M', on='DATE')['PRECTOTCORR'].sum()\n
plt.figure(figsize=(12, 4))\n
monthly_prec.plot(kind='bar')\n
plt.title('Monthly Total PRECTOTCORR (Ethiopia)')\n
plt.ylabel('Precipitation (mm/month)')\n
plt.tight_layout()\n
plt.show()

In [ ]:
corr = df.select_dtypes(include='number').corr(numeric_only=True)\n
plt.figure(figsize=(10, 8))\n
sns.heatmap(corr, cmap='coolwarm', center=0)\n
plt.title('Correlation Heatmap')\n
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=axes[0])\n
axes[0].set_title('T2M vs RH2M')\n
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=axes[1])\n
axes[1].set_title('T2M_RANGE vs WS2M')\n
plt.tight_layout()\n
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))\n
sns.histplot(df['PRECTOTCORR'], bins=50)\n
plt.yscale('log')\n
plt.title('PRECTOTCORR Distribution (log-y scale)')\n
plt.show()\n
\n
plt.figure(figsize=(10, 6))\n
sns.scatterplot(data=df, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(10, 200), alpha=0.5)\n
plt.title('Bubble Chart: T2M vs RH2M (size = PRECTOTCORR)')\n
plt.show()